In [2]:
%%writefile customers.csv
customer_id,customer_name,city,state,age,gender,plan_id,status
101,Rahul Sharma,Hyderabad,Telangana,35,Male,P101,Active
102,Priya Reddy,Bangalore,Karnataka,29,Female,P102,Active
103,Amit Kumar,Mumbai,Maharashtra,42,Male,P103,Inactive
104,Sneha Patel,Chennai,Tamil Nadu,31,Female,P101,Active
105,Farhan Ali,Delhi,Delhi,55,Male,P104,Active
106,Neha Singh,Pune,Maharashtra,38,Female,P102,Active
107,Arjun Verma,Hyderabad,Telangana,26,Male,P103,Inactive
108,Meera Nair,Kochi,Kerala,48,Female,P104,Active
109,Kiran Rao,Bangalore,Karnataka,33,Male,P101,Active
110,Nisha Reddy,Delhi,Delhi,41,Female,P102,Active
111,Ravi Kumar,Mumbai,Maharashtra,45,Male,P105,Active
112,Ayesha Khan,Hyderabad,Telangana,28,Female,,Active

Writing customers.csv


In [3]:
%%writefile usage.csv
usage_id,customer_id,usage_month,data_used_gb,call_minutes,sms_count
1001,101,2026-01,45,900,120
1002,102,2026-01,30,600,80
1003,103,2026-01,12,250,40
1004,104,2026-01,55,1100,150
1005,105,2026-01,75,1500,200
1006,106,2026-01,28,500,60
1007,107,2026-01,10,200,20
1008,108,2026-01,80,1600,250
1009,109,2026-01,48,950,100
1010,110,2026-01,32,700,90
1011,120,2026-01,60,1300,140
1012,101,2026-02,50,1000,130
1013,102,2026-02,34,650,85
1014,104,2026-02,58,1200,160
1015,105,2026-02,,1450,210

Writing usage.csv


In [4]:
%%writefile plans.json
[
{
"plan_id": "P101",
"plan_name": "Smart Basic",
"monthly_fee": 499,
"data_limit_gb": 50,
"features": {
"unlimited_calls": true,
"ott_included": false,
"roaming": "National"
}
},
{

"plan_id": "P102",
"plan_name": "Smart Plus",
"monthly_fee": 799,
"data_limit_gb": 75,
"features": {
"unlimited_calls": true,
"ott_included": true,
"roaming": "National"
}
},
{
"plan_id": "P103",
"plan_name": "Budget Saver",
"monthly_fee": 299,
"data_limit_gb": 25,
"features": {
"unlimited_calls": false,
"ott_included": false,
"roaming": null
}
},
{
"plan_id": "P104",
"plan_name": "Premium Max",
"monthly_fee": 1199,
"data_limit_gb": 100,
"features": {
"unlimited_calls": true,
"ott_included": true,
"roaming": "International"
}
}
]

Writing plans.json


In [5]:
%%writefile payments.csv
payment_id,customer_id,bill_month,amount_paid,payment_mode,payment_status
5001,101,2026-01,499,UPI,Success
5002,102,2026-01,799,Card,Success
5003,103,2026-01,299,Cash,Failed
5004,104,2026-01,499,UPI,Success
5005,105,2026-01,1199,Card,Success
5006,106,2026-01,799,UPI,Success
5007,107,2026-01,299,Cash,Pending
5008,108,2026-01,1199,Card,Success
5009,109,2026-01,499,UPI,Success
5010,110,2026-01,799,UPI,Success
5011,112,2026-01,,UPI,Success
5012,101,2026-02,499,Card,Success
5013,102,2026-02,799,UPI,Success
5014,104,2026-02,499,UPI,Success
5015,105,2026-02,1199,,Pending

Writing payments.csv


In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("Billing Analytics").getOrCreate()

In [7]:
customers_df=spark.read.csv(
    "customers.csv",
    header=True,
    inferSchema=True
)

In [8]:
usage_df=spark.read.csv(
    "usage.csv",
    header=True,
    inferSchema=True
)

In [9]:
payments_df=spark.read.csv(
    "payments.csv",
    header=True,
    inferSchema=True
)

In [10]:
plans_df=spark.read.option(
    "multiline",
    True,
).json("plans.json")

In [11]:
customers_df.printSchema()
usage_df.printSchema()
payments_df.printSchema()
plans_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- plan_id: string (nullable = true)
 |-- status: string (nullable = true)

root
 |-- usage_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- usage_month: timestamp (nullable = true)
 |-- data_used_gb: integer (nullable = true)
 |-- call_minutes: integer (nullable = true)
 |-- sms_count: integer (nullable = true)

root
 |-- payment_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- bill_month: timestamp (nullable = true)
 |-- amount_paid: integer (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- payment_status: string (nullable = true)

root
 |-- data_limit_gb: long (nullable = true)
 |-- features: struct (nullable = true)
 |    |-- ott_included: boolean (nullable = true)
 |

In [12]:
customers_df.count()

12

In [13]:
usage_df.count()

15

In [14]:
payments_df.count()

15

In [15]:
plans_df.count()

4

In [16]:
customers_df.write.mode("overwrite").parquet("bronze/customers")
usage_df.write.mode("overwrite").parquet("bronze/usage")
payments_df.write.mode("overwrite").parquet("bronze/payments")
plans_df.write.mode("overwrite").parquet("bronze/plans")

In [17]:
customers_df.filter(col("plan_id").isNull()).show()

+-----------+-------------+---------+---------+---+------+-------+------+
|customer_id|customer_name|     city|    state|age|gender|plan_id|status|
+-----------+-------------+---------+---------+---+------+-------+------+
|        112|  Ayesha Khan|Hyderabad|Telangana| 28|Female|   NULL|Active|
+-----------+-------------+---------+---------+---+------+-------+------+



In [18]:
payments_df.filter(col("amount_paid").isNull()).show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5011|        112|2026-01-01 00:00:00|       NULL|         UPI|       Success|
+----------+-----------+-------------------+-----------+------------+--------------+



In [19]:
payments_df.filter(col("payment_mode").isNull()).show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5015|        105|2026-02-01 00:00:00|       1199|        NULL|       Pending|
+----------+-----------+-------------------+-----------+------------+--------------+



In [20]:
usage_df.na.fill({"data_used_gb":0}).show()

+--------+-----------+-------------------+------------+------------+---------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|
+--------+-----------+-------------------+------------+------------+---------+
|    1001|        101|2026-01-01 00:00:00|          45|         900|      120|
|    1002|        102|2026-01-01 00:00:00|          30|         600|       80|
|    1003|        103|2026-01-01 00:00:00|          12|         250|       40|
|    1004|        104|2026-01-01 00:00:00|          55|        1100|      150|
|    1005|        105|2026-01-01 00:00:00|          75|        1500|      200|
|    1006|        106|2026-01-01 00:00:00|          28|         500|       60|
|    1007|        107|2026-01-01 00:00:00|          10|         200|       20|
|    1008|        108|2026-01-01 00:00:00|          80|        1600|      250|
|    1009|        109|2026-01-01 00:00:00|          48|         950|      100|
|    1010|        110|2026-01-01 00:00:00|          

In [21]:
payments_df.na.fill({"amount_paid":0}).show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5001|        101|2026-01-01 00:00:00|        499|         UPI|       Success|
|      5002|        102|2026-01-01 00:00:00|        799|        Card|       Success|
|      5003|        103|2026-01-01 00:00:00|        299|        Cash|        Failed|
|      5004|        104|2026-01-01 00:00:00|        499|         UPI|       Success|
|      5005|        105|2026-01-01 00:00:00|       1199|        Card|       Success|
|      5006|        106|2026-01-01 00:00:00|        799|         UPI|       Success|
|      5007|        107|2026-01-01 00:00:00|        299|        Cash|       Pending|
|      5008|        108|2026-01-01 00:00:00|       1199|        Card|       Success|
|      5009|        109|2026-01-01 00:00:00|        499|         

In [22]:
payments_df.na.fill({"payment_mode":"Not Provided"}).show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5001|        101|2026-01-01 00:00:00|        499|         UPI|       Success|
|      5002|        102|2026-01-01 00:00:00|        799|        Card|       Success|
|      5003|        103|2026-01-01 00:00:00|        299|        Cash|        Failed|
|      5004|        104|2026-01-01 00:00:00|        499|         UPI|       Success|
|      5005|        105|2026-01-01 00:00:00|       1199|        Card|       Success|
|      5006|        106|2026-01-01 00:00:00|        799|         UPI|       Success|
|      5007|        107|2026-01-01 00:00:00|        299|        Cash|       Pending|
|      5008|        108|2026-01-01 00:00:00|       1199|        Card|       Success|
|      5009|        109|2026-01-01 00:00:00|        499|         

In [23]:
customers_df.na.fill({"plan_id":"UNKNOWN"}).show()

+-----------+-------------+---------+-----------+---+------+-------+--------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|
+-----------+-------------+---------+-----------+---+------+-------+--------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|  Active|
|        103|   Amit Kumar|   Mumbai|Maharashtra| 42|  Male|   P103|Inactive|
|        104|  Sneha Patel|  Chennai| Tamil Nadu| 31|Female|   P101|  Active|
|        105|   Farhan Ali|    Delhi|      Delhi| 55|  Male|   P104|  Active|
|        106|   Neha Singh|     Pune|Maharashtra| 38|Female|   P102|  Active|
|        107|  Arjun Verma|Hyderabad|  Telangana| 26|  Male|   P103|Inactive|
|        108|   Meera Nair|    Kochi|     Kerala| 48|Female|   P104|  Active|
|        109|    Kiran Rao|Bangalore|  Karnataka| 33|  Male|   P101|  Active|
|        110|  Nisha Reddy|    Delhi|      Delhi| 41|Female|   P

In [24]:
from pyspark.sql.functions import *
customers_df=customers_df.withColumn("data_quality_status",
                                     when(col("plan_id")=="UNKNOWN","Incomplete" ).otherwise("Completed"))

In [25]:
usage_df = usage_df.withColumn(
    "data_quality_status",
    when(col("data_used_gb") == 0, "Incomplete")
    .otherwise("Complete")
)

In [26]:
payments_df = payments_df.withColumn(
    "data_quality_status",
    when(
        (col("amount_paid") == 0) |
        (col("payment_mode") == "Not Provided"),
        "Incomplete"
    ).otherwise("Complete")
)

In [27]:
customers_df.write.mode("overwrite").parquet("silver/customers")
usage_df.write.mode("overwrite").parquet("silver/usage")
payments_df.write.mode("overwrite").parquet("silver/payments")
plans_df.write.mode("overwrite").parquet("silver/plans")

In [28]:
from pyspark.sql.functions import *
plans_flat_df=plans_df.select(
    "plan_id",
    "plan_name",
    "monthly_fee",
    "data_limit_gb",
    col("features.unlimited_calls").alias("unlimited_calls"),
    col("features.ott_included").alias("ott_included"),
    col("features.roaming").alias("roaming")
                              )
plans_flat_df.show()

+-------+------------+-----------+-------------+---------------+------------+-------------+
|plan_id|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|
+-------+------------+-----------+-------------+---------------+------------+-------------+
|   P101| Smart Basic|        499|           50|           true|       false|     National|
|   P102|  Smart Plus|        799|           75|           true|        true|     National|
|   P103|Budget Saver|        299|           25|          false|       false|         NULL|
|   P104| Premium Max|       1199|          100|           true|        true|International|
+-------+------------+-----------+-------------+---------------+------------+-------------+



In [29]:
plans_flat_df.select("plan_id","unlimited_calls").show()

+-------+---------------+
|plan_id|unlimited_calls|
+-------+---------------+
|   P101|           true|
|   P102|           true|
|   P103|          false|
|   P104|           true|
+-------+---------------+



In [30]:
plans_flat_df.select("plan_id","ott_included").show()

+-------+------------+
|plan_id|ott_included|
+-------+------------+
|   P101|       false|
|   P102|        true|
|   P103|       false|
|   P104|        true|
+-------+------------+



In [31]:
plans_flat_df.select("plan_id","roaming").show()

+-------+-------------+
|plan_id|      roaming|
+-------+-------------+
|   P101|     National|
|   P102|     National|
|   P103|         NULL|
|   P104|International|
+-------+-------------+



In [32]:
plans_flat_df=plans_flat_df.na.fill({"roaming":"Not Available"})
plans_flat_df.show()

+-------+------------+-----------+-------------+---------------+------------+-------------+
|plan_id|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|
+-------+------------+-----------+-------------+---------------+------------+-------------+
|   P101| Smart Basic|        499|           50|           true|       false|     National|
|   P102|  Smart Plus|        799|           75|           true|        true|     National|
|   P103|Budget Saver|        299|           25|          false|       false|Not Available|
|   P104| Premium Max|       1199|          100|           true|        true|International|
+-------+------------+-----------+-------------+---------------+------------+-------------+



In [33]:
plans_flat_df.write.mode("overwrite").parquet("silver/plans_flat_df")

In [34]:
customer_plan_df=customers_df.join(
    plans_flat_df,
    "plan_id",
    "left"
)
customer_plan_df.show()

+-------+-----------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+
|plan_id|customer_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|
+-------+-----------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+
|   P101|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|  Active|          Completed| Smart Basic|        499|           50|           true|       false|     National|
|   P102|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|  Active|          Completed|  Smart Plus|        799|           75|           true|        true|     National|
|   P103|        103|   Amit Kumar|   Mumbai|Maharashtra| 42|  Male|Inactive|          Completed|Bud

In [35]:
customer_usage_df = customers_df.join( usage_df, "customer_id", "left" )
customer_usage_df.show()

+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+--------+-------------------+------------+------------+---------+-------------------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|data_quality_status|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+--------+-------------------+------------+------------+---------+-------------------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|          Completed|    1012|2026-02-01 00:00:00|          50|        1000|      130|           Complete|
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|          Completed|    1001|2026-01-01 00:00:00|          45|         900|      120|           Complete|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|  Active|        

In [36]:
customer_payment_df = customers_df.join( payments_df, "customer_id", "left" )
customer_payment_df.show()

+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|          Completed|      5012|2026-02-01 00:00:00|        499|        Card|       Success|           Complete|
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|          Completed|      5001|2026-01-01 00:00:00|        499|         UPI|       Success|           Complete|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Fe

In [37]:
complete_df = customers_df \
.join(plans_flat_df, "plan_id", "left") \
.join(usage_df, "customer_id", "left") \
.join(payments_df, "customer_id", "left")
complete_df.show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------

In [38]:
customers_df.join( plans_flat_df, "plan_id", "left" ).filter( col("plan_name").isNull() ).show()

+-------+-----------+-------------+---------+-----------+---+------+------+-------------------+---------+-----------+-------------+---------------+------------+-------+
|plan_id|customer_id|customer_name|     city|      state|age|gender|status|data_quality_status|plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|roaming|
+-------+-----------+-------------+---------+-----------+---+------+------+-------------------+---------+-----------+-------------+---------------+------------+-------+
|   P105|        111|   Ravi Kumar|   Mumbai|Maharashtra| 45|  Male|Active|          Completed|     NULL|       NULL|         NULL|           NULL|        NULL|   NULL|
|   NULL|        112|  Ayesha Khan|Hyderabad|  Telangana| 28|Female|Active|          Completed|     NULL|       NULL|         NULL|           NULL|        NULL|   NULL|
+-------+-----------+-------------+---------+-----------+---+------+------+-------------------+---------+-----------+-------------+---------------+--------

In [39]:
usage_df.join( customers_df, "customer_id", "left" ).filter( col("customer_name").isNull() ).show()

+-----------+--------+-------------------+------------+------------+---------+-------------------+-------------+----+-----+----+------+-------+------+-------------------+
|customer_id|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|customer_name|city|state| age|gender|plan_id|status|data_quality_status|
+-----------+--------+-------------------+------------+------------+---------+-------------------+-------------+----+-----+----+------+-------+------+-------------------+
|        120|    1011|2026-01-01 00:00:00|          60|        1300|      140|           Complete|         NULL|NULL| NULL|NULL|  NULL|   NULL|  NULL|               NULL|
+-----------+--------+-------------------+------------+------------+---------+-------------------+-------------+----+-----+----+------+-------+------+-------------------+



In [40]:
payments_df.join( customers_df, "customer_id", "left" ).filter( col("customer_name").isNull() ).show()

+-----------+----------+----------+-----------+------------+--------------+-------------------+-------------+----+-----+---+------+-------+------+-------------------+
|customer_id|payment_id|bill_month|amount_paid|payment_mode|payment_status|data_quality_status|customer_name|city|state|age|gender|plan_id|status|data_quality_status|
+-----------+----------+----------+-----------+------------+--------------+-------------------+-------------+----+-----+---+------+-------+------+-------------------+
+-----------+----------+----------+-----------+------------+--------------+-------------------+-------------+----+-----+---+------+-------+------+-------------------+



In [41]:
complete_df = complete_df.withColumn( "usage_category", when(col("data_used_gb") >= 70, "Heavy User") .when(col("data_used_gb") >= 30, "Medium User") .otherwise("Low User") )
complete_df.select( "customer_id", "data_used_gb", "usage_category" ).show()

+-----------+------------+--------------+
|customer_id|data_used_gb|usage_category|
+-----------+------------+--------------+
|        101|          50|   Medium User|
|        101|          50|   Medium User|
|        101|          45|   Medium User|
|        101|          45|   Medium User|
|        102|          34|   Medium User|
|        102|          34|   Medium User|
|        102|          30|   Medium User|
|        102|          30|   Medium User|
|        103|          12|      Low User|
|        104|          58|   Medium User|
|        104|          58|   Medium User|
|        104|          55|   Medium User|
|        104|          55|   Medium User|
|        105|        NULL|      Low User|
|        105|        NULL|      Low User|
|        105|          75|    Heavy User|
|        105|          75|    Heavy User|
|        106|          28|      Low User|
|        107|          10|      Low User|
|        108|          80|    Heavy User|
+-----------+------------+--------

In [42]:
complete_df = complete_df.withColumn( "payment_category", when(col("amount_paid") >= 1000, "High Payment") .when(col("amount_paid") >= 500, "Medium Payment") .otherwise("Low Payment") )
complete_df.select( "customer_id", "amount_paid", "payment_category" ).show()

+-----------+-----------+----------------+
|customer_id|amount_paid|payment_category|
+-----------+-----------+----------------+
|        101|        499|     Low Payment|
|        101|        499|     Low Payment|
|        101|        499|     Low Payment|
|        101|        499|     Low Payment|
|        102|        799|  Medium Payment|
|        102|        799|  Medium Payment|
|        102|        799|  Medium Payment|
|        102|        799|  Medium Payment|
|        103|        299|     Low Payment|
|        104|        499|     Low Payment|
|        104|        499|     Low Payment|
|        104|        499|     Low Payment|
|        104|        499|     Low Payment|
|        105|       1199|    High Payment|
|        105|       1199|    High Payment|
|        105|       1199|    High Payment|
|        105|       1199|    High Payment|
|        106|        799|  Medium Payment|
|        107|        299|     Low Payment|
|        108|       1199|    High Payment|
+----------

In [43]:
complete_df = complete_df.withColumn( "churn_risk", when( (col("status") == "Inactive") | (col("payment_status") != "Success"), "High Risk" ) .when(col("data_used_gb") < 15, "Medium Risk") .otherwise("Low Risk") )
complete_df.select( "customer_id", "status", "payment_status", "data_used_gb", "churn_risk" ).show()

+-----------+--------+--------------+------------+----------+
|customer_id|  status|payment_status|data_used_gb|churn_risk|
+-----------+--------+--------------+------------+----------+
|        101|  Active|       Success|          50|  Low Risk|
|        101|  Active|       Success|          50|  Low Risk|
|        101|  Active|       Success|          45|  Low Risk|
|        101|  Active|       Success|          45|  Low Risk|
|        102|  Active|       Success|          34|  Low Risk|
|        102|  Active|       Success|          34|  Low Risk|
|        102|  Active|       Success|          30|  Low Risk|
|        102|  Active|       Success|          30|  Low Risk|
|        103|Inactive|        Failed|          12| High Risk|
|        104|  Active|       Success|          58|  Low Risk|
|        104|  Active|       Success|          58|  Low Risk|
|        104|  Active|       Success|          55|  Low Risk|
|        104|  Active|       Success|          55|  Low Risk|
|       

In [44]:
complete_df = complete_df.withColumn( "over_usage_gb", col("data_used_gb") - col("data_limit_gb") )
complete_df.select( "customer_id", "data_used_gb", "data_limit_gb", "over_usage_gb" ).show()

+-----------+------------+-------------+-------------+
|customer_id|data_used_gb|data_limit_gb|over_usage_gb|
+-----------+------------+-------------+-------------+
|        101|          50|           50|            0|
|        101|          50|           50|            0|
|        101|          45|           50|           -5|
|        101|          45|           50|           -5|
|        102|          34|           75|          -41|
|        102|          34|           75|          -41|
|        102|          30|           75|          -45|
|        102|          30|           75|          -45|
|        103|          12|           25|          -13|
|        104|          58|           50|            8|
|        104|          58|           50|            8|
|        104|          55|           50|            5|
|        104|          55|           50|            5|
|        105|        NULL|          100|         NULL|
|        105|        NULL|          100|         NULL|
|        1

In [45]:
complete_df = complete_df.withColumn( "over_usage_flag", when(col("over_usage_gb") > 0, "Yes") .otherwise("No") )

In [46]:
customers_df.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|    Kochi|    1|
|  Chennai|    1|
|   Mumbai|    2|
|     Pune|    1|
|    Delhi|    2|
|Hyderabad|    3|
+---------+-----+



In [47]:
customers_df.groupBy("state").count().show()

+-----------+-----+
|      state|count|
+-----------+-----+
|  Karnataka|    2|
|     Kerala|    1|
| Tamil Nadu|    1|
|      Delhi|    2|
|  Telangana|    3|
|Maharashtra|    3|
+-----------+-----+



In [48]:
customers_df.groupBy("plan_id").count().show()

+-------+-----+
|plan_id|count|
+-------+-----+
|   P105|    1|
|   P102|    3|
|   NULL|    1|
|   P103|    2|
|   P104|    2|
|   P101|    3|
+-------+-----+



In [49]:
complete_df.groupBy("usage_category").count().show()

+--------------+-----+
|usage_category|count|
+--------------+-----+
|   Medium User|   14|
|    Heavy User|    3|
|      Low User|    7|
+--------------+-----+



In [50]:
complete_df.groupBy("churn_risk").count().show()

+----------+-----+
|churn_risk|count|
+----------+-----+
|  Low Risk|   20|
| High Risk|    4|
+----------+-----+



In [51]:
complete_df.groupBy("plan_name").agg( sum("data_used_gb").alias("total_data_usage") ).show()

+------------+----------------+
|   plan_name|total_data_usage|
+------------+----------------+
|        NULL|            NULL|
| Smart Basic|             464|
|Budget Saver|              22|
| Premium Max|             230|
|  Smart Plus|             188|
+------------+----------------+



In [52]:
complete_df.groupBy("plan_name").agg( avg("data_used_gb").alias("avg_data_usage") ).show()

+------------+------------------+
|   plan_name|    avg_data_usage|
+------------+------------------+
|        NULL|              NULL|
| Smart Basic| 51.55555555555556|
|Budget Saver|              11.0|
| Premium Max| 76.66666666666667|
|  Smart Plus|31.333333333333332|
+------------+------------------+



In [53]:
complete_df.groupBy("city").agg( sum("call_minutes").alias("total_call_minute") ).show()

+---------+-----------------+
|     city|total_call_minute|
+---------+-----------------+
|Bangalore|             3450|
|    Kochi|             1600|
|  Chennai|             4600|
|   Mumbai|              250|
|     Pune|              500|
|    Delhi|             6600|
|Hyderabad|             4000|
+---------+-----------------+



In [54]:
complete_df.groupBy("state").agg( sum("sms_count").alias("total_sms_count") ).show()

+-----------+---------------+
|      state|total_sms_count|
+-----------+---------------+
|  Karnataka|            430|
|     Kerala|            250|
| Tamil Nadu|            620|
|      Delhi|            910|
|  Telangana|            520|
|Maharashtra|            100|
+-----------+---------------+



In [55]:
complete_df.filter( col("payment_status") == "Success" ).agg( sum("amount_paid") .alias("total_revenue") ).show()

+-------------+
|total_revenue|
+-------------+
|        12882|
+-------------+



In [56]:
complete_df.filter( col("payment_status") == "Success" ).groupBy("city") .agg( sum("amount_paid") .alias("city_revenue") ) .show()

+---------+------------+
|     city|city_revenue|
+---------+------------+
|Bangalore|        3695|
|    Kochi|        1199|
|  Chennai|        1996|
|     Pune|         799|
|    Delhi|        3197|
|Hyderabad|        1996|
+---------+------------+



In [57]:

complete_df.filter( col("payment_status") == "Success" ).groupBy("plan_name") .agg( sum("amount_paid") .alias("plan_revenue") )  .show()

+-----------+------------+
|  plan_name|plan_revenue|
+-----------+------------+
|       NULL|        NULL|
|Smart Basic|        4491|
|Premium Max|        3597|
| Smart Plus|        4794|
+-----------+------------+



In [58]:
complete_df.filter( col("payment_status") == "Success" ).groupBy("payment_mode") \
 .agg( sum("amount_paid") .alias("payment_mode_revenue") ) \
  .show()

+------------+--------------------+
|payment_mode|payment_mode_revenue|
+------------+--------------------+
|        Card|                6193|
|         UPI|                6689|
+------------+--------------------+



In [59]:
complete_df.filter( col("payment_status") == "Success" ).groupBy("plan_name") \
 .agg( sum("amount_paid") .alias("revenue") ) \
 .orderBy(desc("revenue")) \
  .show(1)

+----------+-------+
| plan_name|revenue|
+----------+-------+
|Smart Plus|   4794|
+----------+-------+
only showing top 1 row


In [60]:
complete_df.filter( col("payment_status") == "Success" ).groupBy("city") \
 .agg( sum("amount_paid") .alias("revenue") ) \
 .orderBy(desc("revenue")) \
  .show(1)

+---------+-------+
|     city|revenue|
+---------+-------+
|Bangalore|   3695|
+---------+-------+
only showing top 1 row


In [61]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

In [62]:
usage_window=Window.orderBy(desc("data_used_gb"))
complete_df.withColumn("usage_rank",rank().over(usage_window)).show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+----------+-------------+---------------+----------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category|churn_risk|over_usage_gb|over_usage_flag|usage_rank|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+----------

In [63]:
usage_window=Window.orderBy(desc("amount_paid"))
complete_df.withColumn("payment_rank",rank().over(usage_window)).show()

+-----------+-------+-------------+---------+-----------+---+------+------+-------------------+-----------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+----------+-------------+---------------+------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|status|data_quality_status|  plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category|churn_risk|over_usage_gb|over_usage_flag|payment_rank|
+-----------+-------+-------------+---------+-----------+---+------+------+-------------------+-----------+-----------+---

In [64]:
complete_df.orderBy( desc("data_used_gb") ).show(3)

+-----------+-------+-------------+-----+------+---+------+------+-------------------+-----------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+----------+-------------+---------------+
|customer_id|plan_id|customer_name| city| state|age|gender|status|data_quality_status|  plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category|churn_risk|over_usage_gb|over_usage_flag|
+-----------+-------+-------------+-----+------+---+------+------+-------------------+-----------+-----------+-------------+---------------+------------+-------------

In [65]:
complete_df.groupBy( "customer_id", "customer_name" ).agg( sum("amount_paid").alias("total_payment") ).orderBy( desc("total_payment") ).show(3)

+-----------+-------------+-------------+
|customer_id|customer_name|total_payment|
+-----------+-------------+-------------+
|        105|   Farhan Ali|         4796|
|        102|  Priya Reddy|         3196|
|        101| Rahul Sharma|         1996|
+-----------+-------------+-------------+
only showing top 3 rows


In [66]:
city_window = Window.partitionBy("city").orderBy(desc("data_used_gb"))
complete_df.withColumn( "rn", row_number().over(city_window) ).filter( col("rn") == 1 ).show()


+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+----------+-------------+---------------+---+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category|churn_risk|over_usage_gb|over_usage_flag| rn|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+------------

In [67]:
plan_window = Window.partitionBy("plan_name") \
                    .orderBy(desc("data_used_gb"))

complete_df.withColumn(
    "rn",
    row_number().over(plan_window)
).filter(
    col("rn") == 1
).show()


+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+----------+-------------+---------------+---+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category|churn_risk|over_usage_gb|over_usage_flag| rn|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+------------

In [68]:
month_revenue = complete_df.groupBy( "bill_month" ).agg( sum("amount_paid").alias("monthly_revenue") )
month_window = Window.orderBy("bill_month")
month_revenue.withColumn( "running_revenue", sum("monthly_revenue").over(month_window) ).show()

+-------------------+---------------+---------------+
|         bill_month|monthly_revenue|running_revenue|
+-------------------+---------------+---------------+
|               NULL|           NULL|           NULL|
|2026-01-01 00:00:00|           9886|           9886|
|2026-02-01 00:00:00|           5992|          15878|
+-------------------+---------------+---------------+



In [69]:
customer_window = Window.partitionBy( "customer_id" ).orderBy("usage_month")
complete_df.withColumn( "previous_month_usage", lag("data_used_gb").over(customer_window) ).show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+----------+-------------+---------------+--------------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category|churn_risk|over_usage_gb|over_usage_flag|previous_month_usage|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+---

In [70]:
complete_df.withColumn( "next_month_usage", lead("data_used_gb").over(customer_window) ).show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+----------+-------------+---------------+----------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category|churn_risk|over_usage_gb|over_usage_flag|next_month_usage|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+-----------

In [71]:
usage_growth_df = complete_df.withColumn( "previous_usage", lag("data_used_gb").over(customer_window) )
usage_growth_df.filter( col("data_used_gb") > col("previous_usage") ).show()

+-----------+-------+-------------+---------+----------+---+------+------+-------------------+-----------+-----------+-------------+---------------+------------+--------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+----------+-------------+---------------+--------------+
|customer_id|plan_id|customer_name|     city|     state|age|gender|status|data_quality_status|  plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included| roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category|churn_risk|over_usage_gb|over_usage_flag|previous_usage|
+-----------+-------+-------------+---------+----------+---+------+------+-------------------+-----------+-----------+------------

In [72]:
customers_df.createOrReplaceTempView("customers")
usage_df.createOrReplaceTempView("usage")
payments_df.createOrReplaceTempView("payments")
plans_flat_df.createOrReplaceTempView("plans")
complete_df.createOrReplaceTempView("customer_usage_billing")

In [73]:
spark.sql(""" SELECT city, COUNT(*) AS total_customers FROM customers GROUP BY city """).show()

+---------+---------------+
|     city|total_customers|
+---------+---------------+
|Bangalore|              2|
|    Kochi|              1|
|  Chennai|              1|
|   Mumbai|              2|
|     Pune|              1|
|    Delhi|              2|
|Hyderabad|              3|
+---------+---------------+



In [74]:
spark.sql(""" SELECT city, COUNT(*) AS total_customers FROM customers GROUP BY city """).show()

+---------+---------------+
|     city|total_customers|
+---------+---------------+
|Bangalore|              2|
|    Kochi|              1|
|  Chennai|              1|
|   Mumbai|              2|
|     Pune|              1|
|    Delhi|              2|
|Hyderabad|              3|
+---------+---------------+



In [75]:
spark.sql(""" SELECT p.plan_name, SUM(pay.amount_paid) AS revenue FROM customers c JOIN plans p ON c.plan_id=p.plan_id JOIN payments pay ON c.customer_id=pay.customer_id GROUP BY p.plan_name """).show()

+------------+-------+
|   plan_name|revenue|
+------------+-------+
| Smart Basic|   2495|
|Budget Saver|    598|
| Premium Max|   3597|
|  Smart Plus|   3196|
+------------+-------+



In [76]:
spark.sql(""" SELECT * FROM customer_usage_billing WHERE data_used_gb >= 70 """).show()

+-----------+-------+-------------+-----+------+---+------+------+-------------------+-----------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+----------+-------------+---------------+
|customer_id|plan_id|customer_name| city| state|age|gender|status|data_quality_status|  plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category|churn_risk|over_usage_gb|over_usage_flag|
+-----------+-------+-------------+-----+------+---+------+------+-------------------+-----------+-----------+-------------+---------------+------------+-------------

In [77]:
spark.catalog.listTables()

[Table(name='customer_usage_billing', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='customers', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='payments', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='plans', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='usage', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [78]:
spark.sql(""" SELECT * FROM customer_usage_billing WHERE churn_risk='High Risk' """).show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+----------+-------------+---------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category|churn_risk|over_usage_gb|over_usage_flag|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+------

In [79]:
spark.sql(""" SELECT * FROM customers WHERE plan_id='UNKNOWN' OR plan_id='P105' """).show()

+-----------+-------------+------+-----------+---+------+-------+------+-------------------+
|customer_id|customer_name|  city|      state|age|gender|plan_id|status|data_quality_status|
+-----------+-------------+------+-----------+---+------+-------+------+-------------------+
|        111|   Ravi Kumar|Mumbai|Maharashtra| 45|  Male|   P105|Active|          Completed|
+-----------+-------------+------+-----------+---+------+-------+------+-------------------+



In [80]:
spark.sql(""" SELECT * FROM payments WHERE payment_status IN ('Failed','Pending') """).show()

+----------+-----------+-------------------+-----------+------------+--------------+-------------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|
+----------+-----------+-------------------+-----------+------------+--------------+-------------------+
|      5003|        103|2026-01-01 00:00:00|        299|        Cash|        Failed|           Complete|
|      5007|        107|2026-01-01 00:00:00|        299|        Cash|       Pending|           Complete|
|      5015|        105|2026-02-01 00:00:00|       1199|        NULL|       Pending|           Complete|
+----------+-----------+-------------------+-----------+------------+--------------+-------------------+



In [81]:
spark.sql(""" SELECT customer_id, data_used_gb FROM customer_usage_billing ORDER BY data_used_gb DESC LIMIT 5 """).show()

+-----------+------------+
|customer_id|data_used_gb|
+-----------+------------+
|        108|          80|
|        105|          75|
|        105|          75|
|        104|          58|
|        104|          58|
+-----------+------------+



In [82]:
spark.sql(""" SELECT payment_mode, SUM(amount_paid) AS revenue FROM payments GROUP BY payment_mode """).show()

+------------+-------+
|payment_mode|revenue|
+------------+-------+
|        NULL|   1199|
|        Card|   3696|
|        Cash|    598|
|         UPI|   4393|
+------------+-------+



In [83]:
customers_df = customers_df.withColumnRenamed(
    "data_quality_status",
    "customer_dq_status"
)

usage_df = usage_df.withColumnRenamed(
    "data_quality_status",
    "usage_dq_status"
)

payments_df = payments_df.withColumnRenamed(
    "data_quality_status",
    "payment_dq_status"
)

In [84]:
complete_df = customers_df \
    .join(plans_flat_df, "plan_id", "left") \
    .join(usage_df, "customer_id", "left") \
    .join(payments_df, "customer_id", "left")

In [85]:
complete_df.write.mode("overwrite").parquet("gold/customer_analytics")

In [86]:
print(complete_df.columns)

['customer_id', 'plan_id', 'customer_name', 'city', 'state', 'age', 'gender', 'status', 'customer_dq_status', 'plan_name', 'monthly_fee', 'data_limit_gb', 'unlimited_calls', 'ott_included', 'roaming', 'usage_id', 'usage_month', 'data_used_gb', 'call_minutes', 'sms_count', 'usage_dq_status', 'payment_id', 'bill_month', 'amount_paid', 'payment_mode', 'payment_status', 'payment_dq_status']


In [87]:
gold_df=complete_df
gold_df.write.mode("overwrite") .parquet("gold/customer_analytics")

In [88]:
gold_df.write.mode("overwrite") .partitionBy("usage_month") .parquet("gold/customer_analytics")

In [89]:
%%writefile incremental_usage.csv
usage_id,customer_id,usage_month,data_used_gb,call_minutes,sms_count
1016,101,2026-3,55,1100,140
1017,102,2026-3,40,800,95
1018,104,2026-3,65,1300,170

Writing incremental_usage.csv


In [90]:
incremental_df = spark.read.csv(
    "incremental_usage.csv",
    header=True,
    inferSchema=True
)

In [91]:
incremental_df.write.mode("append").parquet("silver/usage")

In [92]:
usage_df = spark.read.parquet( "silver/usage" )
usage_summary = usage_df.groupBy( "customer_id" ).agg( sum("data_used_gb").alias("total_usage") )

In [93]:
gold_df.write.mode("overwrite") \
.partitionBy("usage_month").parquet("gold/customer_analytics")

In [94]:

print(usage_df.count())

18


In [98]:
from pyspark.sql.functions import countDistinct
plan_report = gold_df.groupBy( "plan_name" ).agg( countDistinct("customer_id").alias("total_customers"), sum("data_used_gb").alias("total_data_usage"), avg("data_used_gb").alias("average_data_usage"), sum("amount_paid").alias("total_revenue") )

In [99]:
city_report = gold_df.groupBy( "city" ).agg( countDistinct("customer_id").alias("total_customers"), sum("amount_paid").alias("total_revenue"), avg("amount_paid").alias("average_payment") )
city_report.show()

+---------+---------------+-------------+---------------+
|     city|total_customers|total_revenue|average_payment|
+---------+---------------+-------------+---------------+
|Bangalore|              2|         3695|          739.0|
|    Kochi|              1|         1199|         1199.0|
|  Chennai|              1|         1996|          499.0|
|   Mumbai|              2|          299|          299.0|
|     Pune|              1|          799|          799.0|
|    Delhi|              2|         5595|         1119.0|
|Hyderabad|              3|         2295|          459.0|
+---------+---------------+-------------+---------------+



In [104]:
over_usage_report = gold_df.select( "customer_id", "customer_name", "plan_name", "data_used_gb", "data_limit_gb")
over_usage_report.show()

+-----------+-------------+------------+------------+-------------+
|customer_id|customer_name|   plan_name|data_used_gb|data_limit_gb|
+-----------+-------------+------------+------------+-------------+
|        101| Rahul Sharma| Smart Basic|          50|           50|
|        101| Rahul Sharma| Smart Basic|          50|           50|
|        101| Rahul Sharma| Smart Basic|          45|           50|
|        101| Rahul Sharma| Smart Basic|          45|           50|
|        102|  Priya Reddy|  Smart Plus|          34|           75|
|        102|  Priya Reddy|  Smart Plus|          34|           75|
|        102|  Priya Reddy|  Smart Plus|          30|           75|
|        102|  Priya Reddy|  Smart Plus|          30|           75|
|        103|   Amit Kumar|Budget Saver|          12|           25|
|        104|  Sneha Patel| Smart Basic|          58|           50|
|        104|  Sneha Patel| Smart Basic|          58|           50|
|        104|  Sneha Patel| Smart Basic|        